In [4]:
!pip install sklearn-pandas pandas scikit-learn

In [5]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn_pandas import DataFrameMapper
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

URL_BASE = "https://raw.githubusercontent.com/FranciscoAmbort/grupo13_castellano_ambort_ia_2026/main/entrega_3/data"

url_train = f"{URL_BASE}/solicitudes_train.csv"
url_test = f"{URL_BASE}/solicitudes_test.csv"
url_comp = f"{URL_BASE}/comportamiento_bancario.csv"

# 2. Carga directa optimizada
df_sol_train = pd.read_csv(url_train)
df_sol_test = pd.read_csv(url_test)
df_comportamiento = pd.read_csv(url_comp)

In [6]:
#intento random forest

#  Random Forest

In [7]:
df_banco_sin_mes = df_comportamiento.drop(columns=['Mes_Referencia'])
comportamiento_resumen = df_banco_sin_mes.groupby('ID_Usuario').mean().reset_index()

df_sol_train_limpio = df_sol_train.drop(columns=['Fecha_Solicitud'])

df_sol_comp = df_sol_train_limpio.merge(comportamiento_resumen, on= 'ID_Usuario', how= 'left')

df_sol_comp = df_sol_comp.drop(columns=['ID_Usuario'])
df_sol_comp.head()

,ID_Solicitud,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Estado_Final,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas
0,1,7900,0.2333,4461,Consumo,1,3567.364783,4.826087
1,2,17400,0.2507,3889,Vivienda,1,6070.002273,8.590909
2,3,8600,0.2368,4035,Vehiculo,0,168.037273,19.590909
3,4,13300,0.2826,4018,Vehiculo,1,7564.528696,3.043478
4,5,6600,0.2314,3357,Otros,1,3504.671304,2.478261


In [8]:
train2, validation2 = train_test_split(df_sol_comp, test_size=0.2, random_state=42)
train2.head()


,ID_Solicitud,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Estado_Final,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas
5045,5046,21200,0.2838,3875,Otros,0,3258.330000,8.250000
26594,26595,12600,0.2646,2958,Negocio,1,4224.257273,5.545455
13655,13656,14000,0.2548,4838,Consumo,0,46.915217,22.391304
30442,30443,8200,0.3224,2009,Vehiculo,1,1322.977391,14.956522
6428,6429,5000,0.2164,6191,Vivienda,1,10678.739565,9.739130


In [9]:
mapper_rf = DataFrameMapper([
    (['Monto_Solicitado'], None),
    (['Tasa_Interes'], None),
    (['Ingreso_Mensual'], None),
    (['Motivo_Prestamo'], [OneHotEncoder()]),
    (['Saldo_Promedio_Mensual'], None),
    (['Dias_Atraso_Otras_Deudas'], None)
])

rf_model_full = Pipeline([
    ('mapper', mapper_rf),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=25,
        criterion='gini',
        random_state=42,
    ))
])

rf_model_full.fit(train2, train2['Estado_Final'])


/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function tosequence is deprecated; tosequence was deprecated in 1.5 and will be removed in 1.7
  warnings.warn(msg, category=FutureWarning)


Pipeline(steps=[('mapper',
                 DataFrameMapper(drop_cols=[],
                                 features=[(['Monto_Solicitado'], None),
                                           (['Tasa_Interes'], None),
                                           (['Ingreso_Mensual'], None),
                                           (['Motivo_Prestamo'],
                                            [OneHotEncoder()]),
                                           (['Saldo_Promedio_Mensual'], None),
                                           (['Dias_Atraso_Otras_Deudas'],
                                            None)])),
                ('classifier',
                 RandomForestClassifier(max_depth=12, min_samples_leaf=25,
                                        n_estimators=300, random_state=42))])

In [10]:

test_eval_rf = df_sol_test.merge(comportamiento_resumen, on='ID_Usuario', how='left').copy()

test_eval_rf['prob_paga'] = rf_model_full.predict_proba(test_eval_rf)[:, 1]

candidatos_rf = test_eval_rf[test_eval_rf['prob_paga'] >= 0.5].sort_values(
    by='prob_paga',
    ascending=False
)

PRESUPUESTO_MAX = 2_000_000
monto_acumulado = 0
ids_aprobados_rf = []

for idx, fila in candidatos_rf.iterrows():
    monto = fila['Monto_Solicitado']
    if monto_acumulado + monto <= PRESUPUESTO_MAX:
        monto_acumulado += monto
        ids_aprobados_rf.append(fila['ID_Solicitud'])
    else:
        continue


test_eval_rf['Decision_Aprobacion'] = test_eval_rf['ID_Solicitud'].isin(ids_aprobados_rf).astype(int)

submission_rf = test_eval_rf[['ID_Solicitud', 'Decision_Aprobacion']]
submission_rf.to_csv('submission_random_forest.csv', index=False)

# 10. Resumen de control
print(f"Total registros: {len(submission_rf)}")
print(f"Créditos aprobados: {submission_rf['Decision_Aprobacion'].sum()}")
print(f"Monto total colocado: ${monto_acumulado:,.2f} USD")


Total registros: 3061
Créditos aprobados: 928
Monto total colocado: $1,999,900.00 USD
